# Exploratory Data Analysis

### Imports

In [ ]:
import pandas as pd
import numpy as np
import altair as alt
from IPython.display import display
import sys

sys.path.append("..")
import src.viz_theme as viz


### Load the Data

In [ ]:
df_clean = pd.read_csv("clean_data.csv")

### Overview

In [ ]:
display(df_clean.head())
display(df_clean.describe(include="all").T)

### Distributions

In [ ]:
numeric_cols = df_clean.select_dtypes(
    include=np.number
).columns.tolist()

categorical_cols = df_clean.select_dtypes(
    include=["object", "category", "string", "bool"]
).columns.tolist()


if numeric_cols:
    (alt.Chart(df_clean)
        .mark_bar()
        .encode(
            alt.X(alt.repeat("repeat"), type="quantitative", bin=alt.Bin(maxbins=30)),
            alt.Y("count()", title="Count"),
        )
        .properties(width=220, height=160)
        .repeat(repeat=numeric_cols, columns=3)
        .properties(title="Distributions of numeric variables")
        .display())

### Boxplot Distributions

In [ ]:
if numeric_cols:
    (alt.Chart(df_clean)
        .mark_boxplot()
        .encode(alt.X(alt.repeat("repeat"), type="quantitative", title=None))
        .properties(width=220, height=50)
        .repeat(repeat=numeric_cols, columns=3)
        .properties(title="Boxplots (whiskers = 1.5 x IQR, orange = median)")
        .display())

### Categorical Distributions (Barplots)

In [ ]:
for col in categorical_cols:
    unique_count = df_clean[col].nunique()

    # Avoid generating unreadable charts for high-cardinality columns
    if unique_count <= 20:
        viz.bar_chart(df_clean, x=col, y="count()", title=f"Distribution of {col}", labels=True)
    else:
        print(f"\n{col}: {unique_count} unique categories")
        display(df_clean[col].value_counts().head(10))

### Correlation Matrix

In [ ]:
if len(numeric_cols) >= 2:
    correlation_matrix = df_clean[numeric_cols].corr(method="pearson")

    corr_long = (correlation_matrix
                 .rename_axis("variable_1").reset_index()
                 .melt(id_vars="variable_1", var_name="variable_2", value_name="correlation"))

    n = len(numeric_cols)
    size = min(viz.CHART_H + 100, max(250, 60 * n))

    heat = alt.Chart(corr_long).encode(
        x=alt.X("variable_2:N", title=None, sort=numeric_cols, axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("variable_1:N", title=None, sort=numeric_cols),
    )
    rects = heat.mark_rect().encode(
        color=alt.Color("correlation:Q",
                        scale=alt.Scale(domain=[-1, 0, 1], range=[viz.ORANGE, "#ffffff", viz.NAVY]),
                        legend=alt.Legend(title="Pearson r"))
    )
    labels = heat.mark_text(fontSize=11).encode(
        text=alt.Text("correlation:Q", format=".2f"),
        color=alt.condition(alt.datum.correlation > 0.6, alt.value("white"), alt.value("#262626")),
    )
    (rects + labels).properties(title="Correlation matrix", width=size, height=size).display()

### Bivariate Analyses

In [ ]:
# Scatter matrix of numeric variables
if len(numeric_cols) >= 2:
    top_numeric = numeric_cols[:6]
    sample = df_clean[top_numeric].dropna()
    sample = sample.sample(min(1000, len(sample)), random_state=42)

    (alt.Chart(sample)
        .mark_circle(size=14, opacity=0.4, color=viz.BLUE)
        .encode(
            alt.X(alt.repeat("column"), type="quantitative", title=None),
            alt.Y(alt.repeat("row"), type="quantitative", title=None),
        )
        .properties(width=110, height=110)
        .repeat(row=top_numeric, column=top_numeric)
        .display())

In [ ]:
for cat_col in categorical_cols:
    if df_clean[cat_col].nunique() <= 15:
        for num_col in numeric_cols[:5]:
            grouped = (
                df_clean.groupby(cat_col, observed=True)[num_col]
                .agg(["count", "mean", "median", "std"])
                .sort_values("mean", ascending=False)
            )
            print(f"\n{num_col} by {cat_col}:")
            display(grouped)

            # Mean by group; top group highlighted in orange
            plot_df = grouped.reset_index()
            top_group = plot_df.iloc[0][cat_col]
            (alt.Chart(plot_df)
                .mark_bar()
                .encode(
                    y=alt.Y(f"{cat_col}:N", sort="-x", title=None),
                    x=alt.X("mean:Q", title=f"Mean {num_col}"),
                    color=alt.condition(alt.datum[cat_col] == top_group,
                                        alt.value(viz.ORANGE), alt.value(viz.NAVY)),
                    tooltip=[cat_col, "count", "mean", "median"],
                )
                .properties(title=f"Mean {num_col} by {cat_col}",
                            width=viz.CHART_W - 150, height=max(100, 28 * len(plot_df)))
                .display())